# Módulo 2 · 04 · Búsqueda de hiperparámetros

Búsqueda aleatoria (Bergstra y Bengio, 2012) con el mismo presupuesto para las dos familias: 16 configuraciones, hasta 15 épocas cada una con parada temprana (paciencia 4) y la misma semilla. Se elige por pérdida de entropía cruzada en validación.

In [ ]:
# parameters
DEMO = False
EJECUTAR = False

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd().resolve().parents[1]
sys.path[:0] = [str(RAIZ), str(RAIZ / "modulo2_distraccion")]

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common.seeds import fijar_semilla
import m2lib, m2exp

fijar_semilla()
cfg = m2exp.config(demo=DEMO)
CLASES = cfg["datos"]["clases"]
ETIQ = [m2lib.NOMBRES[c] for c in CLASES]
FIG, RES = m2lib.FIGURAS, m2lib.RESULTADOS
FIG.mkdir(exist_ok=True); RES.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight"})

In [ ]:
s = "_demo" if DEMO else ""
bus = {}
for fam in ["cnn", "transfer"]:
    if EJECUTAR or not (RES / f"busqueda_{fam}{s}.csv").exists():
        m2exp.busqueda(cfg, fam)
    bus[fam] = pd.read_csv(RES / f"busqueda_{fam}{s}.csv")
    print(fam, len(bus[fam]), "configuraciones,", round(bus[fam]["segundos"].sum() / 60, 1), "minutos de GPU")
bus["transfer"].head(8)

In [ ]:
bus['cnn'].head(8)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (fam, d) in zip(axs, bus.items()):
    d = d.sort_values("config")
    ax.bar(d["config"].astype(str), d["perdida_val"], color=["C3" if i == d["perdida_val"].idxmin() else "C0" for i in d.index])
    ax.set_title({"cnn": "Red desde cero", "transfer": "Red preentrenada con ajuste fino"}[fam])
    ax.set_xlabel("configuración"); ax.set_yscale("log")
axs[0].set_ylabel("pérdida de validación (log)")
fig.savefig(FIG / "busqueda_hiperparametros.png")

La mejor configuración de cada familia se copia en `config/modulo2.yaml` (secciones `cnn` y `transfer`) y se entrena con 30 semillas en el notebook 05.